# 🏆 Notebook 3｜迷你專案：重現課本 1.1 的兩個應用故事

> 對應講義 **Part 1 + Part 5**（知識地圖站 4）
>
> 課本 1.1 講了六個應用故事，我們挑兩個動手做（全部合成資料、沿用 Ch7 學過的特徵武器）：
>
> **任務 A｜輸送帶視覺檢驗**：良品（圓形零件）vs 瑕疵品（沾污/缺角的零件）→ 特徵（面積/圓度/污點數）→ 閾值規則分類。
> **任務 B｜OCR-lite：8 vs 0**：兩個數字形狀差別只在「洞的數量」→ 用 Ch7 提過的「孔洞數特徵」一刀分類。

## 任務 A｜合成良品與瑕疵零件

規則：圓形是良品；疊上一個深色污點、或切掉一角 = 瑕疵。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.draw import disk, polygon
from skimage.measure import label, regionprops

rng = np.random.default_rng(11)
N = 96

def make_part(defect, seed):
    r = np.random.default_rng(seed)
    img = np.zeros((N, N), dtype=int)
    c = (int(r.integers(25, 70)), int(r.integers(25, 70)))   # 圓心
    rad = int(r.integers(20, 30))
    rr0, cc0 = disk(c, rad)
    img[rr0, cc0] = 1
    if defect == 'stain':
        # 污點：貼在零件「邊緣外側」（一定凸出輪廓）＋ 1–2 顆外部塵點（變成額外連通元件）
        ang = r.uniform(0, 2 * np.pi)
        sc = (int(c[0] + (rad - 3) * np.sin(ang)), int(c[1] + (rad - 3) * np.cos(ang)))
        rr2, cc2 = disk(sc, int(r.integers(5, 9)))
        img[np.clip(rr2, 0, N - 1), np.clip(cc2, 0, N - 1)] = 1
        for _ in range(int(r.integers(1, 3))):
            rs, cs = int(r.integers(0, N)), int(r.integers(0, N))
            d3 = disk((rs, cs), int(r.integers(1, 3)))
            img[np.clip(d3[0], 0, N - 1), np.clip(d3[1], 0, N - 1)] = 1
    elif defect == 'notch':                            # 缺角：用多邊形「切」掉一邊
        x0, y0 = c[1], c[0]
        rp, cp = polygon([y0 - rad, y0 - rad, y0 + rad], [x0 - rad, x0 + rad, x0 + rad])
        img[np.clip(rp, 0, N - 1), np.clip(cp, 0, N - 1)] = 0
    return img

good = [make_part(None, s) for s in range(14)]
bad  = [make_part(rng.choice(['stain', 'notch']), s) for s in range(14, 28)]
print('良品', len(good), '件 / 瑕疵', len(bad), '件')

fig, axes = plt.subplots(2, 5, figsize=(12, 4.5))
for i, im in enumerate(good[:5] + bad[:5]):
    ax = axes[i // 5, i % 5]
    ax.imshow(im, cmap='gray'); ax.axis('off')
    ax.set_title('良品' if i < 5 else '瑕疵')
plt.tight_layout(); plt.show()

In [ ]:
# 特徵萃取：面積、圓度、連通元件數（= 污點造成的「多一塊」）
def features(img):
    lab = label(img)
    regs = regionprops(lab)
    main = max(regs, key=lambda r: r.area)             # 主要零件
    area = main.area
    roundness = main.perimeter ** 2 / (4 * np.pi * main.area)   # 圓形≈1（Ch7 武器）
    n_parts = len(regs)                                # 污點 → 元件數變 2
    return area, roundness, n_parts

Fg = np.array([features(im) for im in good])
Fb = np.array([features(im) for im in bad])
print('良品特徵（面積, 圓度, 元件數）前 5 筆:')
print(Fg[:5].round(2))
print('瑕疵特徵（面積, 圓度, 元件數）前 5 筆:')
print(Fb[:5].round(2))

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(Fg[:,1], Fg[:,2], c='#16a34a', s=70, label='良品')
plt.scatter(Fb[:,1], Fb[:,2], c='#dc2626', s=70, label='瑕疵')
plt.xlabel('圓度 γ（=1 越圓）'); plt.ylabel('連通元件數')
plt.title('兩個特徵就把良品/瑕疵分開了！')
plt.legend(); plt.grid(alpha=0.3); plt.show()

# 閾值規則：圓度 ≥ 1.18 或 元件數 ≥ 2 → 瑕疵（1.18 是從散點圖「目測」選的分界——四大問題中「分類器設計」的一種樸素做法）
pred_good = (Fg[:, 1] < 1.18) & (Fg[:, 2] == 1)
pred_bad  = ~((Fb[:, 1] < 1.18) & (Fb[:, 2] == 1))
acc = (pred_good.sum() + pred_bad.sum()) / (len(good) + len(bad))
print(f'閾值規則準確率 = {acc*100:.0f}%  （{len(good)+len(bad)} 件零件）')
print('→ 課本 1.1 的「輸送帶即時檢驗」，本質就是這一頁程式。')

### ✏️ 任務 A 思考題
1. 把 `notch` 的三角切法改成「切得更小」，圓度還分得開嗎？→ 感受「特徵選擇」的邊界在哪。
2. 如果污點剛好疊在零件**內部**（不增加元件數），哪個特徵會失效？該加什麼特徵？（提示：Ch7 的紋理武器）

## 任務 B｜OCR-lite：8 vs 0（孔洞數特徵）

課本 1.1 說「辨識字元是經典應用」；Ch7 知識卡曾提過「孔洞數能區分 8 和 0」。現在真的做一次：

In [ ]:
from skimage.draw import rectangle_perimeter, circle_perimeter

def draw_digit(which, seed=0):
    r = np.random.default_rng(seed)
    img = np.zeros((64, 64), dtype=int)
    if which == 0:
        # 0：單圓環（一個洞）
        rr, cc = disk((32, 32), 24); img[rr, cc] = 1
        rri, cci = disk((32, 32), 13); img[rri, cci] = 0
    else:
        # 8：上下兩個圓環，本體相交（重疊區相連），各自保留一個洞
        for cy in (22, 42):
            rr, cc = disk((cy, 32), 13); img[rr, cc] = 1
            rri, cci = disk((cy, 32), 5);  img[rri, cci] = 0
    # 雜訊筆觸：只放左右兩側（避開中間的洞區），讓任務不至於太簡單
    for _ in range(int(r.integers(0, 6))):
        cc_ = int(r.integers(0, 2)) * 42 + int(r.integers(2, 21))
        img[int(r.integers(2, 62)), cc_] = 1
    return img

fig, axes = plt.subplots(1, 4, figsize=(9, 3))
for i, w in enumerate([0, 0, 8, 8]):
    axes[i].imshow(draw_digit(w, seed=i), cmap='gray'); axes[i].set_title(f'{"8" if w==8 else "0"}（seed={i}）'); axes[i].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# 洞數 = 1 - euler_number（對單一連通物件）
def holes(img):
    lab = label(img)
    regs = regionprops(lab)
    main = max(regs, key=lambda r: r.area)
    return 1 - main.euler_number

preds, truths = [], []
for w in [0, 8]:
    for s in range(20):
        p = holes(draw_digit(w, seed=s + w * 50))
        preds.append(p)
        truths.append(w)
preds, truths = np.array(preds), np.array(truths)

print('各樣本的洞數統計：')
print('  真 8 :', np.unique(preds[truths == 8], return_counts=True))
print('  真 0 :', np.unique(preds[truths == 0], return_counts=True))

rule = preds >= 2                                   # 2 個洞以上就是 8
acc = (rule == (truths == 8)).mean()
print(f'「洞數≥2 = 8」準確率 = {acc*100:.0f}%')
print('→ 課本 1.1 的 OCR 故事，核心就是「找出區分兩類的數字」。這個數字是 Ch7 教你找的。')

## 🏆 迷你專案完成度檢查（對照課本設計循環）

| 步驟 | 任務 A（輸送帶） | 任務 B（OCR） |
|---|---|---|
| 特徵產生 | 面積・圓度・元件數 | 孔洞數（euler） |
| 特徵選擇 | 散點圖選兩維 | 一個就夠 |
| 分類器設計 | 閾值規則 | 閾值規則 |
| 系統評估 | 準確率 | 準確率 |

> 恭喜！你已經走完課本 1.1 + 1.2 的全部核心：**把應用故事變成可量測的特徵，再變成可評估的決策**——這就是模式辨認（以及你之後每一章）的日常。